In [1]:
import math
import networkx as nx
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import gzip
import os
import time

In [2]:
#loading data set
DATA_PATH = "data_set(1).csv.gz"
#take top 20 currencies
CURRENCIES = {"USD", "EUR", "JPY", "GBP", "CNH", "AUD", "CAD", "CHF", "HKD", "SGD", "SEK", "KRW", "NZD", "NOK", "MXN", "INR"}

In [3]:
df = pd.read_csv(DATA_PATH, compression="gzip")
df

,ticker,participant_timestamp,ask_price,bid_price,open,high,low,close,volume
0,C:ZAREUR,1780358399999000000,0,0,0.052708,0.052993,0.052052,0.052717,123113
1,C:SEKGBP,1780358399999000000,0,0,0.080323,0.080415,0.079053,0.079758,42306
2,C:CADHKD,1780358399999000000,0,0,5.678920,5.679485,5.655318,5.662163,121176
3,C:AUDHKD,1780358399999000000,0,0,5.625950,5.635160,5.589400,5.614730,139889
4,C:ZARAED,1780358399999000000,0,0,0.225884,0.226695,0.214094,0.225270,241088
...,...,...,...,...,...,...,...,...,...
29629,C:USDYER,1782950399999000000,0,0,238.250000,238.250000,238.250000,238.250000,3
29630,C:XPFUSD,1782950399999000000,0,0,0.009497,0.009497,0.009479,0.009479,3
29631,C:YERUSD,1782950399999000000,0,0,0.004186,0.004186,0.004185,0.004185,3
29632,C:USDSDG,1782950399999000000,0,0,597.000000,598.000000,597.000000,598.000000,3


In [ ]:
# Clean, filter, and sort the raw dataframe.

def is_symbol_of_interest(symbol: str):
    # First three characters after 'C:'
    currency1 = symbol[2:5]  
    currency2 = symbol[5:8]
    return currency1 in CURRENCIES and currency2 in CURRENCIES

# Sort by the timestamp and convert the timestamp to seconds.
# The timestamp is provided in nanoseconds but has 1 second resolution.
# df = df.sort_values(by="participant_timestamp", kind="stable")
# df["participant_timestamp"] //= int(1e9)

# The exchange always seems to equal 48.

# Remove currencies in which we are not interested.
# df = df[df["ticker"].map(is_symbol_of_interest)]

# OPTIONAL
# Drop duplicate symbols within the same timestamp.
# When this happens, we should keep the last (most up-to-date) prices.
# df = df.drop_duplicates(subset=["ticker", "participant_timestamp"], keep="last")
df.to_csv("filtered_data_set(1).csv")
df


,ticker,participant_timestamp,ask_price,bid_price,open,high,low,close,volume
0,C:ZAREUR,1780358399999000000,0,0,0.052708,0.052993,0.052052,0.052717,123113
1,C:SEKGBP,1780358399999000000,0,0,0.080323,0.080415,0.079053,0.079758,42306
2,C:CADHKD,1780358399999000000,0,0,5.678920,5.679485,5.655318,5.662163,121176
3,C:AUDHKD,1780358399999000000,0,0,5.625950,5.635160,5.589400,5.614730,139889
4,C:ZARAED,1780358399999000000,0,0,0.225884,0.226695,0.214094,0.225270,241088
...,...,...,...,...,...,...,...,...,...
29629,C:USDYER,1782950399999000000,0,0,238.250000,238.250000,238.250000,238.250000,3
29630,C:XPFUSD,1782950399999000000,0,0,0.009497,0.009497,0.009479,0.009479,3
29631,C:YERUSD,1782950399999000000,0,0,0.004186,0.004186,0.004185,0.004185,3
29632,C:USDSDG,1782950399999000000,0,0,597.000000,598.000000,597.000000,598.000000,3


In [5]:
# Add this before filtering to see what's happening
print(f"Original dataset size: {len(df)}")
print(f"Unique tickers in original data: {df['ticker'].nunique()}")
print(f"Sample tickers: {df['ticker'].head(10).tolist()}")

# Check how many match your criteria
matching_tickers = df[df["ticker"].map(is_symbol_of_interest)]
print(f"Rows with currencies of interest: {len(matching_tickers)}")
print(f"Unique matching tickers: {matching_tickers['ticker'].nunique()}")

# Check timestamp distribution
print(f"Unique timestamps: {df['participant_timestamp'].nunique()}")

Original dataset size: 29634
Unique tickers in original data: 1208
Sample tickers: ['C:ZAREUR', 'C:SEKGBP', 'C:CADHKD', 'C:AUDHKD', 'C:ZARAED', 'C:EGPZAR', 'C:HKDTHB', 'C:ZARTWD', 'C:INRZAR', 'C:CADAED']
Rows with currencies of interest: 4850
Unique matching tickers: 194
Unique timestamps: 25
